# Optimización y Guardado del Modelo
## ML Modeling Challenge - Wizeline

Optimización de hiperparámetros con **RandomizedSearchCV** para el modelo ganador (**CatBoost**),
utilizando las features con importancia > 5 obtenidas en la sección 7 del Notebook 2.

**Métrica de selección**: menor **SMAPE de CV**

## 0. Imports y configuración

In [1]:
from pathlib import Path

import joblib
import numpy as np
import plotly.express as px
import polars as pl
import yaml
from catboost import CatBoostRegressor
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import make_scorer
from sklearn.model_selection import KFold, RandomizedSearchCV

from src.pipeline.modeling.train_functions import (
    SEED,
    build_pipeline,
    fugacity_smape_score,
    mape_score,
    run_cross_validation,
    smape_score,
)

N_ITER = 20  # Aumentar para búsqueda más exhaustiva (ej. 100)

print(f'Seed global:              {SEED}')
print(f'Iteraciones de búsqueda:  {N_ITER}')


Seed global:              5000
Iteraciones de búsqueda:  20


## 1. Carga de datos y selección de features

Usamos las features con **importancia > 5** obtenidas en la sección 7 del Notebook 2.
> Actualizar `LIST_FEATURE_SELECTED_V2` con los resultados reales del Notebook 2.

In [2]:
df_train = pl.read_csv('data/training_data.csv')
print(f'Shape: {df_train.shape}')
df_train.head()


Shape: (800, 21)


feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,target
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
432.475954,289.373016,481.3156,358.755566,802.659004,176.761177,72.648102,720.969179,36.327684,83.768878,659.390923,4.385848,516.789458,19.624422,13.16244,42.351948,35.920392,20.755984,13.8143,384.497136,14.364922
517.59625,330.448341,585.920055,22.684031,169.81324,335.60164,284.451476,748.101047,73.701438,358.147215,11.036952,5.563334,2.960064,20.721878,17.740184,1.726915,167.576065,75.492679,2.480979,303.710869,19.984801
189.43935,553.88882,165.83379,202.465927,176.695586,321.155049,407.278389,161.245668,282.269025,221.570899,143.919562,4.536947,581.823741,101.695639,0.653592,486.859084,117.491548,6.420465,20.713314,22.651537,12.944351
237.307878,195.894881,416.752252,468.729031,611.693517,301.411711,241.880655,49.597044,122.396821,13.828319,677.161907,5.518968,45.014729,196.350455,47.638515,411.414213,67.142022,115.630943,8.927957,388.240433,14.79244
602.845256,16.103208,221.759979,345.765574,558.588369,276.704241,408.069566,19.390813,138.769765,146.662193,311.747565,2.136214,133.59043,197.634584,26.278027,111.127557,172.181136,85.869642,30.537857,625.931837,11.802634


In [3]:
with open('config.yaml') as f:
    config = yaml.safe_load(f)

LIST_FEATURE_SELECTED_V2: list[str] = config['features']['selected']

X = df_train.select(LIST_FEATURE_SELECTED_V2).to_numpy()
y = df_train['target'].to_numpy()

print(f'Features seleccionadas ({len(LIST_FEATURE_SELECTED_V2)}): {LIST_FEATURE_SELECTED_V2}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')


Features seleccionadas (5): ['feature_2', 'feature_13', 'feature_9', 'feature_18', 'feature_11']
X shape: (800, 5)
y shape: (800,)


## 2. Espacio de hiperparámetros

Rangos definidos para CatBoost incluyendo los valores **por default** de cada parámetro:

| Parámetro | Default | Rango de búsqueda | Distribución |
|---|---|---|---|
| `iterations` | ~1000 | [100, 1000] | `randint` |
| `learning_rate` | ~0.03 | [0.01, 0.30] | `loguniform` |
| `depth` | 6 | [4, 10] | `randint` |
| `l2_leaf_reg` | 3.0 | [1.0, 10.0] | `uniform` |
| `subsample` | 0.8 | [0.6, 1.0] | `uniform` |

Mejor trial  : #8
SMAPE CV     : 11.1599%
MAPE  CV     : 13.6084%

Mejores hiperparámetros:
  depth: 4
  iterations: 525
  l2_leaf_reg: 9.823412592430358
  learning_rate: 0.03286623364575478
  subsample: 0.6194846045887941

In [4]:
# Los parámetros usan el prefijo 'model__' por la Pipeline (step = 'model')
param_space = {
    'model__iterations': randint(500, 1500),         # default: ~1000
    'model__learning_rate': loguniform(0.029, 0.30),  # default: ~0.03
    'model__depth': randint(5, 11),                  # default: 6
    'model__l2_leaf_reg': uniform(1.0, 9.0),         # default: 3.0  → [1.0, 10.0]
    'model__subsample': uniform(0.6, 0.4),           # default: 0.8  → [0.6, 1.0]
}

print('Espacio de búsqueda:')
for param, dist in param_space.items():
    print(f'  {param}: {dist}')


Espacio de búsqueda:
  model__iterations: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c072b8baf50>
  model__learning_rate: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c072aa2a910>
  model__depth: <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c072b971990>
  model__l2_leaf_reg: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c072aa32110>
  model__subsample: <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c072cfb1510>


## 3. RandomizedSearchCV

Se evalúan `N_ITER` combinaciones aleatorias con **KFold k=5**, optimizando **SMAPE**.  
`refit=False` porque luego reentrenamos manualmente sobre todos los datos.

In [5]:
kf = KFold(n_splits=4, shuffle=True, random_state=SEED)

smape_scorer = make_scorer(smape_score, greater_is_better=False)
mape_scorer = make_scorer(mape_score, greater_is_better=False)

scoring = {
    'smape': smape_scorer,
    'mape': mape_scorer,
}

base_pipeline = build_pipeline(
    CatBoostRegressor(random_seed=SEED, verbose=0)
)

search = RandomizedSearchCV(
    estimator=base_pipeline,
    param_distributions=param_space,
    n_iter=N_ITER,
    cv=kf,
    scoring=scoring,
    refit=False,
    verbose=10,
    random_state=SEED,
    n_jobs=1,  # CatBoost ya paraleliza internamente
)

print(f'Iniciando búsqueda: {N_ITER} combinaciones x {kf.n_splits} folds...')
search.fit(X, y)
print('Búsqueda completada.')


Iniciando búsqueda: 20 combinaciones x 4 folds...
Fitting 4 folds for each of 20 candidates, totalling 80 fits
[CV 1/4; 1/20] START model__depth=8, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694
[CV 1/4; 1/20] END model__depth=8, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694; mape: (test=-12.126) smape: (test=-11.770) total time=   1.4s
[CV 2/4; 1/20] START model__depth=8, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694
[CV 2/4; 1/20] END model__depth=8, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694; mape: (test=-12.763) smape: (test=-11.637) total time=   1.2s
[CV 3/4; 1/20] START model__depth=8, model__iterations=950,

## 4. Mejor configuración de hiperparámetros

In [6]:
# mean_test_smape es negativo (greater_is_better=False) → maximizar = minimizar SMAPE
best_idx = int(np.argmax(search.cv_results_['mean_test_smape']))
best_params_raw: dict = search.cv_results_['params'][best_idx]
best_smape_cv = float(-search.cv_results_['mean_test_smape'][best_idx])
best_mape_cv = float(-search.cv_results_['mean_test_mape'][best_idx])

print(f'Mejor trial  : #{best_idx}')
print(f'SMAPE CV     : {best_smape_cv:.4f}%')
print(f'MAPE  CV     : {best_mape_cv:.4f}%')
print('\nMejores hiperparámetros:')
for k, v in best_params_raw.items():
    clean_k = k.replace('model__', '')
    print(f'  {clean_k}: {v}')


Mejor trial  : #17
SMAPE CV     : 11.2168%
MAPE  CV     : 14.3600%

Mejores hiperparámetros:
  depth: 5
  iterations: 980
  l2_leaf_reg: 9.56569213287667
  learning_rate: 0.038518088621849036
  subsample: 0.8000817293258272


In [7]:
cv_df = (
    pl.DataFrame(
        {
            'smape_cv': (-np.array(search.cv_results_['mean_test_smape'])).tolist(),
            'mape_cv': (-np.array(search.cv_results_['mean_test_mape'])).tolist(),
            'std_smape': search.cv_results_['std_test_smape'].tolist(),
        }
    )
    .with_row_index('trial')
    .sort('smape_cv')
    .head(10)
)

print('Top 10 configuraciones por SMAPE CV (ascendente):')
print(cv_df)


Top 10 configuraciones por SMAPE CV (ascendente):
shape: (10, 4)
┌───────┬───────────┬───────────┬───────────┐
│ trial ┆ smape_cv  ┆ mape_cv   ┆ std_smape │
│ ---   ┆ ---       ┆ ---       ┆ ---       │
│ u32   ┆ f64       ┆ f64       ┆ f64       │
╞═══════╪═══════════╪═══════════╪═══════════╡
│ 17    ┆ 11.21684  ┆ 14.360047 ┆ 0.114324  │
│ 8     ┆ 11.313625 ┆ 14.480627 ┆ 0.224025  │
│ 9     ┆ 11.318088 ┆ 14.361055 ┆ 0.154033  │
│ 10    ┆ 11.327384 ┆ 14.463547 ┆ 0.111377  │
│ 16    ┆ 11.431453 ┆ 14.783858 ┆ 0.18749   │
│ 4     ┆ 11.446541 ┆ 14.658951 ┆ 0.157501  │
│ 18    ┆ 11.459536 ┆ 15.105212 ┆ 0.122299  │
│ 13    ┆ 11.480743 ┆ 14.661323 ┆ 0.350305  │
│ 5     ┆ 11.531402 ┆ 14.666158 ┆ 0.426704  │
│ 2     ┆ 11.610217 ┆ 15.511295 ┆ 0.192182  │
└───────┴───────────┴───────────┴───────────┘


In [8]:
fig = px.bar(
    cv_df.with_columns(pl.col('trial').cast(pl.Utf8)),
    x='smape_cv',
    y='trial',
    orientation='h',
    error_x='std_smape',
    text_auto='.3f',
    color='smape_cv',
    color_continuous_scale='RdYlGn_r',
    title='Top 10 configuraciones — SMAPE CV (menor es mejor)',
    labels={'smape_cv': 'SMAPE CV (%)', 'trial': 'Trial #'},
)
fig.update_traces(textposition='outside')
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    height=420,
)
fig.show()


## 5. Ajuste final y guardado del modelo

Se entrena el modelo con la **mejor configuración** sobre todos los datos de entrenamiento
y se guarda usando **joblib** (más eficiente que pickle para objetos sklearn con arrays NumPy).

In [9]:
# Extraer parámetros del estimador (quitar prefijo 'model__')
catboost_params: dict = {
    k.replace('model__', ''): v for k, v in best_params_raw.items()
}

best_pipeline = build_pipeline(
    CatBoostRegressor(**catboost_params, random_seed=SEED, verbose=0)
)
best_pipeline.fit(X, y)

# Guardar con joblib (más eficiente que pickle para objetos sklearn con arrays NumPy)
model_path = Path(config['model']['artifact_path'])
model_path.parent.mkdir(exist_ok=True)
joblib.dump(best_pipeline, model_path)

print(f'Modelo guardado en : {model_path}')
print(f'Tamaño             : {model_path.stat().st_size / 1024:.1f} KB')


Modelo guardado en : models/catboost_optimized.joblib
Tamaño             : 574.1 KB


## 6. Evaluación del modelo optimizado

Cross-Validation final con la mejor configuración de hiperparámetros.

In [10]:
# Pipeline sin ajustar — run_cross_validation clona internamente antes de cada fold
eval_pipeline = build_pipeline(
    CatBoostRegressor(**catboost_params, random_seed=SEED, verbose=0)
)

metrics = run_cross_validation(eval_pipeline, X, y, n_splits=5, seed=SEED)

rmse_cv = metrics['rmse_cv']
mae_cv = metrics['mae_cv']
mape_cv = metrics['mape_cv']
smape_cv = metrics['smape_cv']
r2_cv = metrics['r2_cv']
fugacity_smape_cv = metrics['fugacity_smape_cv']

print('Métricas CV — CatBoost optimizado:')
print(f'  RMSE              : {rmse_cv:.4f}')
print(f'  MAE               : {mae_cv:.4f}')
print(f'  MAPE              : {mape_cv:.4f}%')
print(f'  SMAPE             : {smape_cv:.4f}%')
print(f'  R²                : {r2_cv:.4f}')
print(f'  Fugacity SMAPE    : {fugacity_smape_cv:.4f}%')


Métricas CV — CatBoost optimizado:
  RMSE              : 1.7074
  MAE               : 1.3730
  MAPE              : 14.0308%
  SMAPE             : 11.2442%
  R²                : 0.8847
  Fugacity SMAPE    : 23.5000%


In [11]:
df_metrics = pl.DataFrame(
    {
        'metric': ['RMSE', 'MAE', 'MAPE (%)', 'SMAPE (%)', 'R²', 'Fugacity SMAPE (%)'],
        'value': [rmse_cv, mae_cv, mape_cv, smape_cv, r2_cv, fugacity_smape_cv],
    }
)

fig = px.bar(
    df_metrics,
    x='metric',
    y='value',
    text_auto='.4f',
    color='value',
    color_continuous_scale='Blues',
    title='Métricas CV — CatBoost Optimizado',
    labels={'metric': 'Métrica', 'value': 'Valor'},
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=450)
fig.show()
